### Search Engine with Tools and Agents

In [1]:
## Arxiv- for research paper

## tools creation
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper


In [ ]:
## wikipedia tool

api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250) # type: ignore
wiki= WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name


'wikipedia'

In [3]:
## Arxiv tool

api_arxiv_wiki=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250) # type: ignore
arxiv= ArxivQueryRun(api_wrapper=api_arxiv_wiki)
arxiv.name

'arxiv'

In [4]:
tools=[wiki,arxiv]

In [6]:
## Custom tools [RAG Based]

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [7]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs= loader.load()

documents=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever= vectordb.as_retriever()
retriever


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002354A452120>, search_kwargs={})

In [ ]:
## retriever to custom tool
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith")
retriever_tool.name


'langsmith-search'

In [11]:
tools=[wiki,arxiv,retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Shub\\LangchainProject\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000002357D6BA7A0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000002357D6BA840>)]

In [ ]:
## Run all this tool with Agent and llm models

## tools , llm model -> Agents executor

from dotenv import load_dotenv
from langchain_groq import ChatGroq
import openai
import os
load_dotenv()
groq_api_key= os.getenv("GROQ_API_KEY") # type: ignore
openai.api_key  = os.getenv("OPENAI_API_KEY") # type: ignore

llm = ChatGroq(api_key=groq_api_key,model="llama-3.1-8b-instant") # type: ignore

python-dotenv could not parse statement starting at line 6


In [35]:
## Prompt template

from langsmith import Client

# Initialize the modern client
client = Client()

# The modern client natively accepts the safety parameter
prompt = client.pull_prompt(
    "hwchase17/openai-functions-agent", 
    dangerously_pull_public_prompt=True
)

print(prompt.messages)

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]


In [36]:
from langchain_classic.agents import create_openai_tools_agent
agent  = create_openai_tools_agent(llm,tools,prompt)
agent


RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [37]:
## agent executer

from langchain_classic.agents import AgentExecutor

agent_exe = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_exe

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [39]:
agent_exe.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before exploring further.LangSmith is a framework-agnostic platform for b

{'input': 'Tell me about Langsmith',
 'output': "It seems that Langsmith has a vast amount of information, but it seems that the information is quite repetitive. I'll provide a summary of Langsmith's benefits:\n\nLangsmith provides a platform for building, debugging, and deploying AI agents and LLM applications. It offers features such as observability, evaluation, prompt engineering, and deployment, which can help improve the reliability and quality of AI applications. Langsmith also provides a visual interface for designing, testing, and refining applications, and it supports multiple frameworks and providers.\n\nSome benefits of using Langsmith include:\n\n* Improved debugging and reliability\n* Enhanced quality and consistency\n* Increased productivity\n* Scalability\n* Flexibility\n* Security and compliance\n\nKeep in mind that the information provided is based on the Langsmith documentation, and it may not be exhaustive. If you have specific questions or need further clarificatio

In [49]:
agent_exe.invoke({"input":"What is Machine Learning"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine Learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `wikipedia` with `{'query': 'Machine learning algorithm'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `langsmith-search` with `{'query': 'Machine Learning'}`


​More ways to build
FleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage trace

{'input': 'What is Machine Learning',
 'output': 'It seems like the provided information is more focused on LangSmith, which is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. It seems like the best course of action would be to provide more information about machine learning in general, rather than focusing on LangSmith. Here is a revised response:\n\n Machine learning (ML) is a field of study in artificial intelligence that involves the use of algorithms and statistical models to enable machines to learn from data and make decisions or predictions without being explicitly programmed. ML involves several key concepts, including:\n\n1. Supervised learning: This type of learning involves training a model on labeled data, where the correct output is provided for each input.\n2. Unsupervised learning: This type of learning involves training a model on unlabeled data, where the model must find patterns or structure in the data.\n3. Reinfo

In [92]:
agent_exe.invoke({"input":"What's the paper 1706.03762 about ?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`




AttributeError: 'Search' object has no attribute 'results'